## 문제 1-1 : 기본 Chain 만들기 - AI 요리사
### 문제 설명
사용자가 재료를 입력하면 그 재료로 만들 수 있는 요리를 추천해주는 간단한 AI 요리사를 만들어보세요.

### 요구사항
1. PromptTemplate을 사용하여 프롬프트 작성
2. 사용자가 입력한 재료를 받아서 만들 수 있는 요리 추천
3. ChatOpenAI 모델 사용
4. StrOutputParser로 결과를 문자열로 출력
5. LCEL(|) 문법을 사용하여 체인 연결

### 구현 조건
* **입력** : 재료명 (예: "토마토, 양파, 치즈")
* **출력** : 추천 요리와 간단한 레시피

### 예상 실행 결과
```text
입력: "계란, 밥, 김치"
출력: 
계란과 밥, 김치로 만들 수 있는 요리를 추천드립니다!

추천 요리: 김치볶음밥
재료: 계란, 밥, 김치
조리법:
1. 팬에 기름을 두르고 김치를 볶아주세요
2. 밥을 넣고 함께 볶아주세요
3. 계란을 풀어서 넣고 잘 섞어주세요
...
```

### 1. 환경 구성
#### 1) 라이브러리 설치

In [43]:
# uv add python-dotenv langchain langchain-openai

#### 2) OpenAI 인증키 설정

In [44]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


In [45]:
import langchain
print(langchain.__version__)

1.4.3


In [46]:
from langchain_openai import ChatOpenAI

# OpenAI 공통 설정 — 모델/온도 변경은 이곳만 수정
OPENAI_MODEL = "gpt-4o-mini"  # 대안: "gpt-4o"
TEMPERATURE = 0.7


def get_llm(model: str = OPENAI_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """OpenAI API를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=OPENAI_API_KEY,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()

### 2. PromptTemplate 사용

In [47]:
from langchain_core.prompts import PromptTemplate

template = """당신은 AI 요리사입니다.
사용자가 가지고 있는 재료로 만들 수 있는 요리를 하나 추천하고, 간단한 레시피를 알려주세요.

재료: {ingredients}
요구사항: 
- 가능한 한 사용자가 가진 재료만 사용하세요. (단, 기본 양념은 사용 가능)
- 레시피는 초보자도 따라할 수 있도록 단계별로 간단하게 작성하세요.
- 아래 출력 형식을 지켜서 한국어로 답변하세요.

출력 형식:
(입력 받은 재료를 언급하며 요리를 추천드린다는 한 줄 인사말)

추천 요리: (요리 이름)
재료: (사용하는 재료 이름)
조리법:
1. ...
2. ...
3. ...
"""

prompt = PromptTemplate.from_template(template)

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['ingredients'] input_types={} partial_variables={} template='당신은 AI 요리사입니다.\n사용자가 가지고 있는 재료로 만들 수 있는 요리를 하나 추천하고, 간단한 레시피를 알려주세요.\n\n재료: {ingredients}\n요구사항: \n- 가능한 한 사용자가 가진 재료만 사용하세요. (단, 기본 양념은 사용 가능)\n- 레시피는 초보자도 따라할 수 있도록 단계별로 간단하게 작성하세요.\n- 아래 출력 형식을 지켜서 한국어로 답변하세요.\n\n출력 형식:\n(입력 받은 재료를 언급하며 요리를 추천드린다는 한 줄 인사말)\n\n추천 요리: (요리 이름)\n재료: (사용하는 재료 이름)\n조리법:\n1. ...\n2. ...\n3. ...\n'


### 3. PromptTemplate + LLM(invoke()) + StrOutputParser

In [49]:
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser

# 3. chain의 invoke 호출
ingredients = "계란, 밥, 김치"
result = chain.invoke({"ingredients": ingredients})

print(f'입력: "{ingredients}"')
print(f'출력: {result}')

입력: "계란, 밥, 김치"
출력: 계란, 밥, 김치로 맛있는 요리를 추천드려요!

추천 요리: 김치 계란 볶음밥  
재료: 계란, 밥, 김치  
조리법:  
1. 팬에 기름을 두르고 중불로 예열합니다.  
2. 김치를 작게 썰어 팬에 넣고, 1-2분 정도 볶아서 김치의 향을 내줍니다.  
3. 밥을 넣고 김치와 잘 섞이도록 2-3분 볶습니다.  
4. 밥이 따뜻해지면 계란을 깨서 넣고, 재료들이 잘 섞이도록 저어가며 볶습니다.  
5. 모든 재료가 잘 섞이고 계란이 익으면, 소금이나 후추로 간을 맞추고 완성합니다.  

맛있게 드세요!


### 4. 사용자가 입력한 재료를 받아서 실행

In [50]:
# 사용자에게 입력(input) 받아서 실행 
ingredients = input("재료를 입력하세요 (예: 토마토, 양파, 치즈): ")
result = chain.invoke({"ingredients": ingredients})

print(f'입력: "{ingredients}"')
print(f'출력: {result}')

입력: "토마토, 양파, 치즈"
출력: 토마토, 양파, 치즈로 간단한 요리를 만들어보세요!

추천 요리: 토마토 양파 치즈 샐러드  
재료: 토마토, 양파, 치즈, 올리브유, 소금, 후추 (기본 양념)  
조리법:  
1. 토마토와 양파를 깨끗이 씻고, 토마토는 적당한 크기로 썰고, 양파는 얇게 채 썬다.  
2. 큰 그릇에 썰어놓은 토마토와 양파를 넣는다.  
3. 치즈를 원하는 크기로 잘라서 그릇에 추가한다.  
4. 올리브유와 소금, 후추를 뿌려 잘 섞어준다.  
5. 완성된 샐러드를 접시에 담고, 원하시면 추가로 후추를 뿌려서 즐긴다.  

맛있게 드세요!


## 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템
### 문제 설명
사용자가 특정 도시나 국가를 입력하면, 해당 지역의 대표적인 관광 명소를 추천하고 그 추천받은 관광 명소에 대한 상세 정보(역사, 특징, 방문 팁 등)를 알려주는 2단계 체인을 구현해 보세요.

### 요구사항
1. **1단계 체인**: 여행지(도시/국가)를 입력받아 대표 명소 1가지 추천
2. **2단계 체인**: 추천받은 명소의 상세 정보 제공
3. ChatPromptTemplate 사용 : “system” 과 “user” 메시지를 지정합니다.
4. 두 체인을 LCEL로 연결
5. 각 단계의 결과를 모두 출력하여 과정 확인

### 구현 조건
* **입력**: 여행지 (예: "로마", "뉴욕", "도쿄")
* **1단계 출력**: 추천 명소 이름
* **2단계 출력**: 명소에 대한 상세 정보 (역사, 특징, 방문 팁 등)

### 예상 실행 결과
```text
입력: "로마" 1단계 결과: "콜로세움" 
추천 2단계 결과: 명소: 콜로세움 
역사:
로마 시대에 건설된 거대한 원형 경기장입니다.
... (이하 역사 설명)
특징:
검투사들의 시합이 열리던 장소입니다.
... (이하 특징 설명)
방문 팁:
미리 온라인으로 표를 예매하면 좋습니다.
... (이하 방문 팁)
```

### 1. ChatPromptTemplate (system + user)

In [51]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1: 사용자가 입력한 여행지(도시/국가)의 대표 명소 1가지 추천
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 전문가입니다. 사용자가 입력한 여행지의 대표적인 관광 명소를 딱 1곳만 추천합니다. \
     설명, 문장부호, 따옴표 없이 명소 이름만 한국어로 답변하세요."),
    ("user", "{destination}에서 꼭 가봐야 할 대표 관광 명소 1곳을 추천해주세요."),
])

# Step 2: 추천받은 명소의 상세 정보(역사, 특징, 방문 팁) 제공 
prompt2 = ChatPromptTemplate.from_messages([
    ("system", """당신은 여행 가이드입니다. 사용자가 알려준 명소의 상세 정보를 아래 출력 형식에 맞춰 한국어로 알려주세요.
     각 항목은 번호를 붙여 2~3개씩 간단하게 작성하세요.
     
     출력 형식:
     명소: (명소 이름)

     역사:
     1. ...
     2. ...

     특징:
     1. ...
     2. ...

     방문 팁:
     1. ...
     2. ...
     """),
    ("user", "{spot}에 대한 상세 정보(역사, 특징, 방문 팁)를 알려주세요."),
])

print(prompt1)
print(prompt2)

input_variables=['destination'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 여행 전문가입니다. 사용자가 입력한 여행지의 대표적인 관광 명소를 딱 1곳만 추천합니다.      설명, 문장부호, 따옴표 없이 명소 이름만 한국어로 답변하세요.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['destination'], input_types={}, partial_variables={}, template='{destination}에서 꼭 가봐야 할 대표 관광 명소 1곳을 추천해주세요.'), additional_kwargs={})]
input_variables=['spot'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 여행 가이드입니다. 사용자가 알려준 명소의 상세 정보를 아래 출력 형식에 맞춰 한국어로 알려주세요.\n     각 항목은 번호를 붙여 2~3개씩 간단하게 작성하세요.\n\n     출력 형식:\n     명소: (명소 이름)\n\n     역사:\n     1. ...\n     2. ...\n\n     특징:\n     1. ...\n     2. ...\n\n     방문 팁:\n     1. ...\n     2. ...\n     '), additional_kwargs={}), HumanMessagePr

### 2. 1단계 체인: 여행지 → 대표 명소 추천

In [52]:
from langchain_core.output_parsers import StrOutputParser

# 체인 1: 명소 추천 (입력: 여행지 → 출력: 명소 이름)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 여행지에 따라 명소 추천
destination = "로마"
spot = chain1.invoke({"destination": destination})  

print(f'입력: "{destination}" 1단계 결과: "{spot}"')  

입력: "로마" 1단계 결과: "콜로세움"


### 3. 2단계 체인: 추천 명소 → 상세 정보 (LCEL로 두 체인 연결)

In [53]:
from langchain_core.runnables import RunnablePassthrough

# 체인 2: 상세 정보 (입력: 명소 이름 → 출력: 역사, 특징, 방문 팁)
chain2 = prompt2 | llm | StrOutputParser()

# 두 체인을 LCEL로 연결
multi_chain = (
    {"spot": chain1} 
    | RunnablePassthrough.assign(info=chain2)  
)

# Step 2: 여행지의 대표 명소 추천 및 상세 정보
destination = "로마"
response = multi_chain.invoke({"destination": destination})

print(f'입력: "{destination}"')
print(f'1단계 결과: "{response["spot"]}" 추천')
print(f'2단계 결과: {response["info"]}')

입력: "로마"
1단계 결과: "콜로세움" 추천
2단계 결과: 명소: 콜로세움

역사:
1. 콜로세움은 기원후 70년부터 80년 사이에 로마 황제 베스파시아누스에 의해 건설되었으며, 그 후 그의 아들 티투스에 의해 완공되었습니다.
2. 고대 로마의 검투사 경기와 다양한 대중 오락이 진행된 장소로, 최대 50,000명의 관중을 수용할 수 있었습니다.

특징:
1. 원형 경기장 형태로, 지름 약 188미터, 높이 48미터로 로마의 상징적인 건축물 중 하나입니다.
2. 콜로세움의 외벽은 대리석으로 장식되어 있으며, 다양한 건축 양식을 혼합한 독특한 디자인이 특징입니다.

방문 팁:
1. 사전 예약을 통해 입장권을 구매하면 긴 대기 시간을 줄일 수 있습니다.
2. 가이드 투어에 참여하면 콜로세움의 역사와 숨겨진 이야기들을 더욱 깊이 이해할 수 있습니다.


## 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용
### 뉴스 키워드 추출기
### 문제 설명
FewShotPromptTemplate을 사용하여 뉴스 기사에서 핵심 키워드 3개를 추출하는 시스템을 구현해보세요. 주어진 예시들을 참고하여 일관된 형식으로 키워드를 추출해야 합니다.

### 요구사항
1. FewShotPromptTemplate 사용
2. 최소 3개의 예시(examples) 포함
3. 뉴스 텍스트에서 핵심 키워드 3개 추출
4. 일관된 출력 형식 유지
5. 다양한 분야의 뉴스로 테스트

### 예시 데이터
```python
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    # 추가 예시 필요
]
```
### 테스트 뉴스
"제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."


### 예상 출력 형식
키워드: 제미나이, 구글AI, 개발자

### 구현 힌트
```python
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate, 
    ChatPromptTemplate
)

# 1. 예시 프롬프트 (대화형)
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])
examples = [
    {
        "news": "삼성전자가 AI 가속기를 출시할 예정이다.",
        "keywords": "삼성전자, AI가속기, 출시"
    },
    {
        "news": "WHO가 국제 협력의 중요성을 강조했다.",
        "keywords": "WHO, 국제협력, 보건"
    }
]

# 2. Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 3. 최종 프롬프트
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요."),
    few_shot_prompt,
    ("human", "{input}")
])
```

### 1. FewShotChatMessagePromptTemplate

In [54]:
from langchain_core.prompts import (
    FewShotChatMessagePromptTemplate, 
    ChatPromptTemplate
)

# 1. 예시 프롬프트 (대화형)
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    {
        "news": "한국은행이 금융통화위원회를 열고 기준금리를 현 수준에서 동결하기로 결정했다. 물가 상승세가 둔화되고 있지만 가계부채 증가와 환율 변동성에 대한 우려가 여전히 크다는 점이 동결 배경으로 꼽힌다.",
        "keywords": "한국은행, 기준금리, 가계부채"
    },
    {
        "news": "한국 축구 국가대표팀이 월드컵 아시아 최종예선 경기에서 승리를 거두며 본선 진출에 한 걸음 더 다가섰다. 감독은 경기 후 인터뷰에서 선수들의 조직력과 집중력이 승리의 원동력이었다고 밝혔다.",
        "keywords": "축구대표팀, 월드컵, 최종예선"
    },
    {
        "news": "정부가 온실가스 감축 목표 달성을 위해 재생에너지 발전 비중을 대폭 확대하는 계획을 발표했다. 태양광과 해상풍력 설비를 늘리고 석탄 화력발전소는 단계적으로 폐쇄해 탄소중립을 앞당긴다는 방침이다.",
        "keywords": "온실가스, 재생에너지, 탄소중립"
    },
]

# 2. Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 3. 최종 프롬프트
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요."),
    few_shot_prompt,
    ("human", "{input}")
])

# 모델 생성 및 체인 구성
chain = final_prompt | llm

# 실행
results = chain.batch([{"input": example["news"]} for example in examples])
for result in results:
    print(result.content)

키워드: 삼성전자, AI 가속기, 엔비디아
키워드: 세계보건기구, 건강 위기, 국제 협력
키워드: 한국은행, 기준금리, 가계부채
키워드: 축구대표팀, 월드컵, 아시아 최종예선
키워드: 온실가스, 재생에너지, 탄소중립


In [56]:
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0)
output_parser = StrOutputParser()

# chain 생성 (LCEL)
chain = final_prompt | llm | output_parser

# chain의 invoke 호출
# 테스트 뉴스 
test_news = "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."

result = chain.invoke({"input": test_news})
print(result)

키워드: 제미나이 2.0, 구글 AI, 멀티모달 입력
